In [3]:
import threading
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()  # loads .env file

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def call_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    return response.choices[0].message.content

# 🔁 Shared Memory
class SharedMemory:
    def __init__(self):
        self.data = {}
        self.lock = threading.Lock()

    def write(self, agent, key, value):
        with self.lock:
            if agent not in self.data:
                self.data[agent] = {}
            self.data[agent][key] = value

    def read(self, agent, key):
        with self.lock:
            return self.data.get(agent, {}).get(key, "")

    def read_all(self):
        with self.lock:
            return self.data.copy()



# 🔵 Search Agent (LLM-based)
class SearchAgent:
    name = "search"

    def run(self, goal, memory):
        print("🔍 SearchAgent working...")
        prompt = f"""
        You are a research assistant.
        Collect key facts about: {goal}
        Include:
        - Market size
        - Growth trends
        - Government policies in India
        """
        result = call_llm(prompt)
        memory.write(self.name, "data", result)


# 🟠 Analyst Agent
class AnalystAgent:
    name = "analyst"

    def run(self, goal, memory):
        print("📊 AnalystAgent analyzing...")
        search_data = memory.read("search", "data")

        prompt = f"""
        Analyze the following EV market data:

        {search_data}

        Provide:
        - Key insights
        - Opportunities
        - Risks
        """
        analysis = call_llm(prompt)
        memory.write(self.name, "analysis", analysis)


# 🟢 Writer Agent
class WriterAgent:
    name = "writer"

    def run(self, goal, memory):
        print("✍️ WriterAgent drafting...")
        search_data = memory.read("search", "data")
        analysis = memory.read("analyst", "analysis")

        prompt = f"""
        Write a professional market report on:

        {goal}

        Use:
        Data: {search_data}
        Analysis: {analysis}

        Format:
        - Introduction
        - Market Trends
        - Insights
        - Risks
        - Conclusion
        """
        report = call_llm(prompt)
        memory.write(self.name, "report", report)


# 🟧 QA Agent
class QAAgent:
    name = "qa"

    def run(self, goal, memory):
        print("✅ QAAgent reviewing...")
        report = memory.read("writer", "report")

        prompt = f"""
        Review the following report for:
        - Accuracy
        - Clarity
        - Professional tone

        Improve if needed:

        {report}
        """
        final_report = call_llm(prompt)
        memory.write(self.name, "final_report", final_report)


# 🟦 Orchestrator
class Orchestrator:
    def __init__(self, agents):
        self.agents = agents
        self.memory = SharedMemory()

    def run(self, goal):
        for agent in self.agents:
            agent.run(goal, self.memory)

        return self.memory.read("qa", "final_report")


# 🚀 Run
if __name__ == "__main__":
    agents = [
        SearchAgent(),
        AnalystAgent(),
        WriterAgent(),
        QAAgent()
    ]

    orchestrator = Orchestrator(agents)

    goal = "EV industry trends in India for 2025"
    final_report = orchestrator.run(goal)

    print("\n📄 FINAL REPORT:\n")
    print(final_report)

🔍 SearchAgent working...
📊 AnalystAgent analyzing...
✍️ WriterAgent drafting...
✅ QAAgent reviewing...

📄 FINAL REPORT:

**Accuracy:**

The report appears to be well-researched and accurate in its analysis of the Indian EV market. However, there are a few areas where accuracy could be improved:

1. **Market size projection:** The report projects the Indian EV market to reach ₹ 2.5 lakh crore by 2025, growing at a CAGR of 30-40% per annum. While this is a reasonable projection, it would be helpful to provide a more detailed breakdown of the market size and growth rate.
2. **Charging infrastructure growth:** The report states that the number of public charging stations in India is expected to increase from 3,000 in 2023 to 10,000 by 2025. This is a reasonable projection, but it would be helpful to provide more context on the current state of charging infrastructure in India and the challenges associated with expanding it.
3. **Government incentives:** The report mentions that the Indian 